# Latin Line Break (`<lb/>`) Detector

This notebook demonstrates how to use the Flair model `mschonhardt/latin-contextual-lb-detector`.
It detects if a line break `<lb/>` in Latin text acts as a 
**word break** (hyphenation) or a **separate word** (space).

Model can be found on [Hugging Face](https://huggingface.co/mschonhardt/latin-contextual-lb-detector) and [Zenodo](https://doi.org/10.5281/zenodo.18390269).


![](https://zenodo.org/badge/DOI/10.5281/zenodo.18390269.svg)


## Setup Environment

In [62]:
# Install flair if running in Colab or if not installed locally
# !pip install -q flair
# Flair 0.15.1 recommended

import flair
print(f"Flair version: {flair.__version__}")

print("Environment ready.")

Flair version: 0.15.1
Environment ready.


## Load the Model from Hugging Face

In [63]:
from flair.models import SequenceTagger
from flair.data import Sentence

print("Loading model: mschonhardt/latin-contextual-lb-detector ...")
tagger = SequenceTagger.load('mschonhardt/latin-contextual-lb-detector')
print("Model loaded successfully!")

Loading model: mschonhardt/latin-contextual-lb-detector ...
2026-03-11 20:52:27,964 SequenceTagger predicts: Dictionary with 5 tags: O, NB, WB, <START>, <STOP>
Model loaded successfully!


### Prediction Logic
We tokenize by **regex** so that `<lb/>` elements (including those with attributes like `<lb type="nb" break="no" xml:id="..."/>`) are treated as a single token, regardless of internal spaces. Adjacent tags like `<lb xml:id="..."/><pb .../>` are also split correctly.


In [ ]:
import re

# Matches a complete self-closing XML tag (with any attributes, including spaces) OR a plain word
# TODO: How to distinguish `<hi>Y</hi>O` from `memoria <choice><abbr>Spartanorũ</abbr><expan>Spartanorum</expan></choice>`?
_TOKEN_RE = re.compile(r'<[^>]+>|[^\s<]+')

def predict_line_breaks(text_input):
    # Tokenize by regex: each XML element (e.g. <lb type="nb" break="no" xml:id="..."/>)
    # is captured as ONE token, regardless of internal spaces.
    # Adjacent tags like <lb xml:id="..."/><pb .../> are also split correctly.
    token_list = _TOKEN_RE.findall(text_input)
    if not token_list:
        return []

    print(f"Tokenized input: {token_list}")

    # As the model expects `<lb/>` as the token for line breaks, we need to ensure that all line break tags are in this format.
    # This means we will replace any `<lb .../>` with `<lb/>` for the purpose of prediction, but we will keep the original tokens for output.
    token_list_for_prediction = [token if not token.startswith('<lb') else '<lb/>' for token in token_list]

    # make sure both token lists have the same length
    assert len(token_list) == len(token_list_for_prediction), "Token lists must be of the same length"
    # make sure indices of <lb> tokens match in both lists
    for i, token in enumerate(token_list):
        if token.startswith('<lb'):
            assert token_list_for_prediction[i] == '<lb/>', f"Token at index {i} should be <lb/> for prediction"

    # Create Flair Sentence
    sentence = Sentence(token_list_for_prediction, use_tokenizer=False)

    # Predict
    tagger.predict(sentence)

    # Extract Results
    results = []
    for i, token in enumerate(sentence):
        if token.text == '<lb/>':
            tag = token.get_label().value
            score = token.get_label().score

            # Map tags to human-readable meanings
            meaning = "Separate Words (Space)" if tag == "NB" else "Split Word (Join)"

            results.append({
                "text": token_list[i],   # full original element with all attributes
                "prediction": tag,
                "confidence": score,
                "meaning": meaning
            })
        else:
            results.append({
                "text": token.text,
                "prediction": "WORD",
                "confidence": 1.0,
                "meaning": "Regular Word"
            })

    return results


### Run Inference

In [94]:
import json

# Example 1: A split words of lines

# text1 = "impleatur, et ideo sequatur absolutio, quod post sententiam possit ite <lb/> rato agi. Ego considero, quod terminus offerendi est vsque ad sententiam"
text1 = "alter<lb xml:id=\"W0022-00-0178-lb-0001\"/>præſidiis terreſtribus præerat) impetra<lb type=\"nb\" break=\"no\" rendition=\"#hyphen\" xml:id=\"W0022-00-0178-lb-0002\"/>runt. magnum enim, quod adſerebant,<lb xml:id=\"W0022-00-0178-lb-0003\"/>videbatur, &amp; Cæſarem id ſummè ſciebãt<lb xml:id=\"W0022-00-0178-lb-0004\"/>cupere."
# text1 = "alter<lb/>præſidiis terreſtribus præerat) impetra<lb/>runt. magnum enim, quod adſerebant,<lb/>videbatur, &amp; Cæſarem id ſummè ſciebãt<lb/>cupere."
print(f"Input: {text1}")
print(f"Prediction: {json.dumps(predict_line_breaks(text1), indent=2)}")

print("-" * 30)

# Example 2: Separate words of lines
# text2 = "vt d. l. si rem. §. fi. Item considero, quod secundus creditor bene habet <lb/> hypothecariam, licet prior ei pręferatur, tamen sua hypothecaria non"
text2 = "lacedæmoniorum tyranno,<lb xml:id=\"W0022-00-0178-lb-0014\"/>ſex menſium indutias dedit, vt interim"
# text2 = "lacedæmoniorum tyranno,<lb/>ſex menſium indutias dedit, vt interim"
print(f"Input: {text2}")
print(f"Prediction: {json.dumps(predict_line_breaks(text2), indent=2)}")


Input: alter<lb xml:id="W0022-00-0178-lb-0001"/>præſidiis terreſtribus præerat) impetra<lb type="nb" break="no" rendition="#hyphen" xml:id="W0022-00-0178-lb-0002"/>runt. magnum enim, quod adſerebant,<lb xml:id="W0022-00-0178-lb-0003"/>videbatur, &amp; Cæſarem id ſummè ſciebãt<lb xml:id="W0022-00-0178-lb-0004"/>cupere.
Tokenized input: ['alter<lb', 'xml:id="W0022-00-0178-lb-0001"/>præſidiis', 'terreſtribus', 'præerat)', 'impetra<lb', 'type="nb"', 'break="no"', 'rendition="#hyphen"', 'xml:id="W0022-00-0178-lb-0002"/>runt.', 'magnum', 'enim,', 'quod', 'adſerebant,<lb', 'xml:id="W0022-00-0178-lb-0003"/>videbatur,', '&amp;', 'Cæſarem', 'id', 'ſummè', 'ſciebãt<lb', 'xml:id="W0022-00-0178-lb-0004"/>cupere.']
Prediction: [
  {
    "text": "alter<lb",
    "prediction": "WORD",
    "confidence": 1.0,
    "meaning": "Regular Word"
  },
  {
    "text": "xml:id=\"W0022-00-0178-lb-0001\"/>pr\u00e6\u017fidiis",
    "prediction": "WORD",
    "confidence": 1.0,
    "meaning": "Regular Word"
  },
  {
  

## Integrate in postprocessing Pipeline
Now we can use prediction to join or split lines depending on the actual workflow.

In [95]:
def set_break_attr(lb_tag: str, value: str) -> str:
    """Set or replace the break=\"...\" attribute on an <lb.../> element."""
    # Remove any existing break=\"...\" or break='...'
    tag = re.sub(r'''\s*break=["'][^"']*["']''', '', lb_tag)
    # Insert break=\"value\" just before the closing />
    tag = re.sub(r'\s*/>$', f' break="{value}"/>', tag)
    return tag

def remove_break_attr(lb_tag: str) -> str:
    """Remove the break=\"...\" attribute from an <lb.../> element."""
    return re.sub(r'''\s*break=["'][^"']*["']''', '', lb_tag)

def set_cert_attr(lb_tag: str, confidence: float) -> str:
    """Set or replace the cert=\"...\" attribute on an <lb.../> element."""
    conf_value = f"{confidence:.2f}"
    # Remove any existing cert=\"...\"
    tag = re.sub(r'''\s*cert=["'][^"']*["']''', '', lb_tag)
    # Insert cert=\"confidence\" just before the closing />
    tag = re.sub(r'\s*/>$', f' cert="{conf_value}"/>', tag)
    return tag

def set_resp_attr(lb_tag: str, editor: str) -> str:
    """Set or replace the resp=\"...\" attribute on an <lb.../> element."""
    # Remove any existing resp=\"...\"
    tag = re.sub(r'''\s*resp=["'][^"']*["']''', '', lb_tag)
    # Insert resp=\"meaning\" just before the closing />
    tag = re.sub(r'\s*/>$', f' resp="{editor}"/>', tag)
    return tag

In [96]:
def recreate_sentence(text_input):
    # Get predictions for line breaks
    predictions = predict_line_breaks(text_input)

    # Reconstruct sentence based on predictions
    reconstructed = []
    for token in predictions:
        if token["text"].startswith('<lb'):
            if token["prediction"] == "WB":  # WB = Split Word (Join) ⇒ set break="true" and cert="{confidence}"
                reconstructed.append(set_break_attr(set_cert_attr(set_resp_attr(remove_break_attr(token["text"]), "ML"), token["confidence"]), "true"))
            else:  # NB = Separate Words (Space) ⇒ remove break attribute and set cert="{confidence}"
                reconstructed.append(set_break_attr(set_cert_attr(set_resp_attr(remove_break_attr(token["text"]), "ML"), token["confidence"]), "false"))
        # elif token.startswith('<'):
        #     pass  # skip pb/cb/other structural tags
        else:
            reconstructed.append(token["text"])

    # Join and clean up spacing
    result = " ".join(reconstructed)
    # Remove spaces before all break elements
    result = re.sub(r'\s+<lb', '<lb', result)
    # Remove spaces after non-breaking break elements (above, make sure that @break attribute is the last attribute before />)
    result = re.sub(r'break="true"\s*/>\s+', 'break="no"/>', result)
    # Fix (invert) meaning of break="false" to break="yes" for TEI compliance
    result = re.sub(r'break="false"\s*/>\s+', 'break="yes"/>', result)
    # Normalize spaces (in case multiple spaces were introduced)
    result = re.sub(r' {2,}', ' ', result)
    return result.strip()

# Test with the existing examples
print("Reconstructed text 1:")
print(recreate_sentence(text1))

print("\n" + "="*50 + "\n")

print("Reconstructed text 2:")
print(recreate_sentence(text2))


Reconstructed text 1:
Tokenized input: ['alter<lb', 'xml:id="W0022-00-0178-lb-0001"/>præſidiis', 'terreſtribus', 'præerat)', 'impetra<lb', 'type="nb"', 'break="no"', 'rendition="#hyphen"', 'xml:id="W0022-00-0178-lb-0002"/>runt.', 'magnum', 'enim,', 'quod', 'adſerebant,<lb', 'xml:id="W0022-00-0178-lb-0003"/>videbatur,', '&amp;', 'Cæſarem', 'id', 'ſummè', 'ſciebãt<lb', 'xml:id="W0022-00-0178-lb-0004"/>cupere.']
alter<lb xml:id="W0022-00-0178-lb-0001"/>præſidiis terreſtribus præerat) impetra<lb type="nb" break="no" rendition="#hyphen" xml:id="W0022-00-0178-lb-0002"/>runt. magnum enim, quod adſerebant,<lb xml:id="W0022-00-0178-lb-0003"/>videbatur, &amp; Cæſarem id ſummè ſciebãt<lb xml:id="W0022-00-0178-lb-0004"/>cupere.


Reconstructed text 2:
Tokenized input: ['lacedæmoniorum', 'tyranno,<lb', 'xml:id="W0022-00-0178-lb-0014"/>ſex', 'menſium', 'indutias', 'dedit,', 'vt', 'interim']
lacedæmoniorum tyranno,<lb xml:id="W0022-00-0178-lb-0014"/>ſex menſium indutias dedit, vt interim


Now, we process full TEI XML files:

1. Open File
2. Find (sequence of) innermost `<text></text>` elements, and for each,
3. Predict `<lb/>`s
4. Save under with `.lb.xml` extension

In [ ]:
# Select file to process with file picker
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

file_picker = widgets.FileUpload(accept='.xml', multiple=False)
display(file_picker)

FileUpload(value=(), accept='.xml', description='Upload')

```python
# After the user selects a file, we can read its content and process it.
with open(Path(file_picker.value[0]['name']), 'r', encoding='utf-8') as f:
    xml_content = f.read()
    print("Original XML content:")
    print(xml_content[:500])  # print first 500 chars for preview
    # Find innermost `<text></text>` content if it exists
    p = re.compile(r'<text[^>]*>(.*?)</text>', re.DOTALL)
    match = p.search(xml_content)
    if match:
        text_content = match.group(1)
        print("\nExtracted <text> content:")
        print(text_content[:500])  # print first 500 chars for preview

        print("\nProcessing...\n")
        predicted_xml = recreate_sentence(text_content)

        print("Reconstructed XML content:")
        print(predicted_xml[:500])  # print first 500 chars for preview

        # Optionally, you could replace the original <text> content with the predicted XML in the full document here.
        xml_content_with_predictions = xml_content.replace(text_content, predicted_xml)

        # Save the modified XML to a new file
        output_path = Path(file_picker.value[0]['metadata']['name']).with_name(Path(file_picker.value[0]['metadata']['name']).stem + '.lb.xml')
        with open(output_path, 'w', encoding='utf-8') as f_out:
            f_out.write(xml_content_with_predictions)
        print(f"\nModified XML saved to: {output_path}")

    else:
        print("No <text> element found in the XML.")
```

Alternatively, use LXML to traverse XML and feed the document element-wise to the function.

In [ ]:
import lxml.etree as ET
# from typing import List, Dict, Any

def process_document(input_file, output_file):
    print(f"Processing document: {input_file}")
    tree = ET.parse(input_file)
    root = tree.getroot()
    texts = [t for t in root.iter('{http://www.tei-c.org/ns/1.0}text')]

    lb_elements = [l for t in texts for l in t.iter('{http://www.tei-c.org/ns/1.0}lb')]
    paragraphs = [p for t in texts for p in t.iter('{http://www.tei-c.org/ns/1.0}p')]  # Adjust based on your TEI structure
    # TODO: add heads, linegroups, items etc. as needed
    print(f"Processing {len(lb_elements)} lb elements in {len(paragraphs)} paragraphs...")
    
    for para in paragraphs:
        lbs_in_para = [l for l in para.iter('{http://www.tei-c.org/ns/1.0}lb')]
        print(f"Processing paragraph {para.get('{http://www.w3.org/XML/1998/namespace}id')} with {len(lbs_in_para)} lb elements")
        print(f"Original paragraph text: {ET.tostring(para, encoding='unicode')[:500]}...")  # Print first 500 chars for preview

        # Process in smaller chunks
        # for i in range(0, len(lbs_in_para), self.max_chunk_size):
        #    chunk = lbs_in_para[i:i + self.max_chunk_size]
        #    self.process_chunk(chunk, doc)

        new_para = recreate_sentence(ET.tostring(para).decode())
        print(f"Modified paragraph text: {new_para[:500]}...")  # Print first 500 chars for preview
        para = ET.fromstring(new_para)  # Replace original paragraph with modified one

    print(f"Writing modified document to: {output_file}")
    tree.write(output_file, encoding='utf-8', xml_declaration=True)

# Usage
process_document(Path(file_picker.value[0]['name']), Path(file_picker.value[0]['name']).with_name(Path(file_picker.value[0]['name']).stem + '.lb.xml'))


Processing document: W0116_008.noglyph.xml
Processing 15703 lb elements in 1657 paragraphs...
Processing paragraph W0116-00-0003-pa-03eb with 16 lb elements
Original paragraph text: <p xmlns="http://www.tei-c.org/ns/1.0" xmlns:t="http://www.tei-c.org/ns/tite/1.0" xmlns:tei="http://www.tei-c.org/ns/1.0" xmlns:xi="http://www.w3.org/2001/XInclude" xml:id="W0116-00-0003-pa-03eb"><lb xml:id="W0116-00-0003-lb-0002"/><foreign xml:lang="es"><hi rendition="#initCaps">Y</hi>O Iuan Aluarez del Marmol eſcriuano<lb xml:id="W0116-00-0003-lb-0003"/>de Camara de ſu Mageſtad, de los que en<lb xml:id="W0116-00-0003-lb-0004"/>el ſu Conſejo reſiden; doy fee, que auien<lb xml:id="W0116-00-0003-...
Tokenized input: ['<p xmlns="http://www.tei-c.org/ns/1.0" xmlns:t="http://www.tei-c.org/ns/tite/1.0" xmlns:tei="http://www.tei-c.org/ns/1.0" xmlns:xi="http://www.w3.org/2001/XInclude" xml:id="W0116-00-0003-pa-03eb">', '<lb xml:id="W0116-00-0003-lb-0002"/>', '<foreign xml:lang="es">', '<hi rendition="#initCaps">',

KeyboardInterrupt: 